SUPERSEDED by notebooks/12_replay_engine.ipynb and the pitwall/ package. Kept for history. Known flaw: confounded compound/age pace table (see 12).

In [ ]:
from dataclasses import dataclass, replace
from typing import Optional

@dataclass(frozen=True)
class DriverState:
    driver_number: str
    position: int
    compound: str
    tyre_age: int
    pit_stops: int = 0
    pit_in_lap: Optional[int] = None
    retired: bool = False


@dataclass(frozen=True)
class RaceConditions:
    lap_number: int
    rainfall: Optional[float]
    grip_condition: str
    weather_state: str
    uncertainty_label: str = "observed_or_assumed"


def age_tyres(state):
    return replace(state, tyre_age=state.tyre_age + 1)


def apply_pit_stop(state, new_compound, current_lap):
    return replace(
        state,
        compound=new_compound,
        tyre_age=0,
        pit_stops=state.pit_stops + 1,
        pit_in_lap=current_lap,
    )

In [ ]:
def summarize_pit_loss(pit_events):
    if "pit_loss_seconds" not in pit_events.columns:
        raise KeyError(
            "Expected pit_loss_seconds. Derive this from pit events and "
            "a validated no-stop pace estimate before calling this function."
        )

    losses = pd.to_numeric(
        pit_events["pit_loss_seconds"], errors="coerce"
    ).dropna()

    if len(losses) < 5:
        raise ValueError(
            "Too few valid pit-loss observations for a stable distribution."
        )

    return {
        "n": int(len(losses)),
        "median_seconds": float(losses.median()),
        "q10_seconds": float(losses.quantile(0.10)),
        "q90_seconds": float(losses.quantile(0.90)),
        "std_seconds": float(losses.std()),
    }

In [ ]:
def build_weather_scenarios(
    current_lap,
    final_lap,
    current_weather_state,
):
    """
    Illustrative scenario generator.

    The scenarios below are hypotheses, not learned probabilities.
    Replace probabilities with calibrated estimates from historical
    weather transitions when sufficient data are available.
    """
    remaining_laps = final_lap - current_lap

    scenarios = [
        {
            "name": "Weather persists",
            "probability": 1 / 3,
            "remaining_laps": remaining_laps,
            "assumption": f"Continue from {current_weather_state}",
        },
        {
            "name": "Weather improves",
            "probability": 1 / 3,
            "remaining_laps": remaining_laps,
            "assumption": "Rain decreases and track may dry",
        },
        {
            "name": "Weather worsens",
            "probability": 1 / 3,
            "remaining_laps": remaining_laps,
            "assumption": "Rain intensity increases",
        },
    ]

    return pd.DataFrame(scenarios)

In [ ]:
def estimate_pace_with_support(
    pace_table,
    compound,
    weather_state,
    min_observations=10,
):
    """
    pace_table must contain:
        compound, weather_state, lap_duration

    Returns the observed median only when the comparison group has
    sufficient observations. This is a diagnostic baseline, not a
    causal tyre-performance estimate.
    """
    required = {"compound", "weather_state", "lap_duration"}
    missing = required - set(pace_table.columns)

    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    subset = pace_table.loc[
        (pace_table["compound"] == compound)
        & (pace_table["weather_state"] == weather_state),
        "lap_duration",
    ].dropna()

    if len(subset) < min_observations:
        return {
            "compound": compound,
            "weather_state": weather_state,
            "n": len(subset),
            "median_seconds": np.nan,
            "supported": False,
            "reason": "Insufficient comparable observations",
        }

    return {
        "compound": compound,
        "weather_state": weather_state,
        "n": len(subset),
        "median_seconds": float(subset.median()),
        "supported": True,
        "reason": (
            "Observed association only; weather and compound effects "
            "may be confounded"
        ),
    }

In [ ]:
def compare_strategy_scenarios(scenario_results):
    """
    Required columns:
        strategy
        scenario
        probability
        projected_total_time

    Projected times must include all modeled remaining laps and pit losses.
    Scenario probabilities must sum to 1 for each strategy comparison.
    """
    required = {
        "strategy",
        "scenario",
        "probability",
        "projected_total_time",
    }
    missing = required - set(scenario_results.columns)

    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    df = scenario_results.copy()

    for col in ["probability", "projected_total_time"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[["probability", "projected_total_time"]].isna().any().any():
        raise ValueError("Missing probabilities or projected times.")

    if (df["probability"] < 0).any():
        raise ValueError("Scenario probabilities cannot be negative.")

    if (df["projected_total_time"] <= 0).any():
        raise ValueError("Projected total times must be positive.")

    prob_sums = df.groupby("strategy")["probability"].sum()

    if not np.allclose(prob_sums.to_numpy(), 1.0, atol=1e-6):
        raise ValueError(
            "Scenario probabilities must sum to 1 for each strategy."
        )

    summary = (
        df.assign(
            weighted_time=lambda x:
                x["probability"] * x["projected_total_time"]
        )
        .groupby("strategy")
        .agg(
            expected_total_time=("weighted_time", "sum"),
            worst_scenario_time=("projected_total_time", "max"),
            scenario_count=("scenario", "nunique"),
        )
        .sort_values("expected_total_time")
        .reset_index()
    )

    return summary

In [ ]:

import numpy as np
import pandas as pd


def evaluate_strategy_table(
    scenarios: pd.DataFrame,
    probability_tolerance: float = 1e-6,
) -> pd.DataFrame:
    """
    Compare strategies using probability-weighted projected race time.

    Required columns:
        strategy
        scenario
        probability
        projected_total_time

    Lower expected projected time is better.
    Results are conditional on the supplied scenarios and assumptions.
    """

    required = {
        "strategy",
        "scenario",
        "probability",
        "projected_total_time",
    }

    missing = required - set(scenarios.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    df = scenarios[list(required)].copy()

    if df.empty:
        raise ValueError("No strategy scenarios supplied.")

    if df[["strategy", "scenario"]].isna().any().any():
        raise ValueError("Strategy and scenario cannot be missing.")

    if df.duplicated(["strategy", "scenario"]).any():
        raise ValueError(
            "Each strategy must have at most one row per scenario."
        )

    for col in ["probability", "projected_total_time"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if not np.isfinite(
        df[["probability", "projected_total_time"]].to_numpy()
    ).all():
        raise ValueError(
            "Probabilities and projected times must be finite numbers."
        )

    if (df["probability"] < 0).any():
        raise ValueError("Probabilities cannot be negative.")

    if (df["projected_total_time"] <= 0).any():
        raise ValueError("Projected total times must be positive.")

    # Each strategy must be evaluated over a probability distribution
    # that sums to 1.
    probability_sums = df.groupby("strategy")["probability"].sum()

    invalid = probability_sums[
        (probability_sums - 1.0).abs() > probability_tolerance
    ]

    if not invalid.empty:
        raise ValueError(
            "Scenario probabilities must sum to 1 for every strategy. "
            f"Invalid sums: {invalid.to_dict()}"
        )

    df["weighted_time"] = (
        df["probability"] * df["projected_total_time"]
    )

    summary = (
        df.groupby("strategy")
        .agg(
            expected_projected_time=("weighted_time", "sum"),
            best_case_time=("projected_total_time", "min"),
            worst_case_time=("projected_total_time", "max"),
            scenario_count=("scenario", "nunique"),
        )
        .reset_index()
    )

    summary["scenario_range"] = (
        summary["worst_case_time"] - summary["best_case_time"]
    )

    summary = summary.sort_values(
        "expected_projected_time"
    ).reset_index(drop=True)

    summary.insert(
        0,
        "rank_by_expected_time",
        np.arange(1, len(summary) + 1),
    )

    return summary


# ---------------------------------------------------------
# TEST ONLY: synthetic example, not real Silverstone output
# ---------------------------------------------------------

synthetic_scenarios = pd.DataFrame([
    {
        "strategy": "Stay out",
        "scenario": "Dry",
        "probability": 0.7,
        "projected_total_time": 5400.0,
    },
    {
        "strategy": "Stay out",
        "scenario": "Rain",
        "probability": 0.3,
        "projected_total_time": 5460.0,
    },
    {
        "strategy": "Pit for Intermediates",
        "scenario": "Dry",
        "probability": 0.7,
        "projected_total_time": 5430.0,
    },
    {
        "strategy": "Pit for Intermediates",
        "scenario": "Rain",
        "probability": 0.3,
        "projected_total_time": 5390.0,
    },
])

test_result = evaluate_strategy_table(synthetic_scenarios)

print("SYNTHETIC TEST ONLY")
display(test_result.round(2))


SYNTHETIC TEST ONLY


,rank_by_expected_time,strategy,expected_projected_time,best_case_time,worst_case_time,scenario_count,scenario_range
0,1,Pit for Intermediates,5418.0,5390.0,5430.0,2,40.0
1,2,Stay out,5418.0,5400.0,5460.0,2,60.0


In [ ]:

import inspect

functions_to_check = [
    "summarize_pit_loss",
    "build_weather_scenarios",
    "estimate_pace_with_support",
    "compare_strategy_scenarios",
    "apply_pit_stop",
    "evaluate_strategy_table",
]

for function_name in functions_to_check:
    function_obj = globals().get(function_name)

    print("\n" + "=" * 75)
    print(function_name)

    if not callable(function_obj):
        print("NOT FOUND: function is not currently defined in this notebook.")
        continue

    print("Signature:", inspect.signature(function_obj))

    try:
        print(inspect.getsource(function_obj))
    except (OSError, TypeError):
        print("Source unavailable. Signature shown above.")



summarize_pit_loss
Signature: (pit_events)
def summarize_pit_loss(pit_events):
    if "pit_loss_seconds" not in pit_events.columns:
        raise KeyError(
            "Expected pit_loss_seconds. Derive this from pit events and "
            "a validated no-stop pace estimate before calling this function."
        )

    losses = pd.to_numeric(
        pit_events["pit_loss_seconds"], errors="coerce"
    ).dropna()

    if len(losses) < 5:
        raise ValueError(
            "Too few valid pit-loss observations for a stable distribution."
        )

    return {
        "n": int(len(losses)),
        "median_seconds": float(losses.median()),
        "q10_seconds": float(losses.quantile(0.10)),
        "q90_seconds": float(losses.quantile(0.90)),
        "std_seconds": float(losses.std()),
    }


build_weather_scenarios
Signature: (current_lap, final_lap, current_weather_state)
def build_weather_scenarios(
    current_lap,
    final_lap,
    current_weather_state,
):
    """
    Illu

In [ ]:
# Notebook 11: Controlled strategy-evaluator test
# These are synthetic values, NOT real Silverstone predictions.

import pandas as pd
import numpy as np

test_scenarios = pd.DataFrame([
    {
        "strategy": "Stay out",
        "scenario": "Dry",
        "probability": 0.7,
        "projected_total_time": 5400.0,
    },
    {
        "strategy": "Stay out",
        "scenario": "Rain",
        "probability": 0.3,
        "projected_total_time": 5460.0,
    },
    {
        "strategy": "Pit for Intermediates",
        "scenario": "Dry",
        "probability": 0.7,
        "projected_total_time": 5430.0,
    },
    {
        "strategy": "Pit for Intermediates",
        "scenario": "Rain",
        "probability": 0.3,
        "projected_total_time": 5390.0,
    },
])

test_results = evaluate_strategy_table(test_scenarios)

display(test_results)

# Verify the expected times and ranking.
expected_times = {
    "Stay out": 5418.0,
    "Pit for Intermediates": 5418.0,
}

for _, row in test_results.iterrows():
    assert np.isclose(
        row["expected_projected_time"],
        expected_times[row["strategy"]],
    )

assert len(test_results) == 2
assert test_results["rank_by_expected_time"].tolist() == [1, 2]

print("PASS: evaluator calculates probability-weighted times correctly.")
print("NOTE: The two strategies tie. No real race recommendation is implied.")

,rank_by_expected_time,strategy,expected_projected_time,best_case_time,worst_case_time,scenario_count,scenario_range
0,1,Pit for Intermediates,5418.0,5390.0,5430.0,2,40.0
1,2,Stay out,5418.0,5400.0,5460.0,2,60.0


PASS: evaluator calculates probability-weighted times correctly.
NOTE: The two strategies tie. No real race recommendation is implied.


In [ ]:
# Notebook 11, Cell 23
# Derive exploratory pit-loss observations from actual pit events.
#
# IMPORTANT:
# This is a diagnostic estimate, not a causal measurement.
# We do not assume a fixed pit-loss value.

from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed/9558")

lap_performance_path = DATA_DIR / "lap_performance.parquet"
pit_events_path = DATA_DIR / "historical_pit_events.parquet"
laps_path = DATA_DIR / "laps.parquet"

for path in [lap_performance_path, pit_events_path, laps_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path.resolve()}")

lap_performance = pd.read_parquet(lap_performance_path)
pit_events = pd.read_parquet(pit_events_path)
laps = pd.read_parquet(laps_path)

print("Lap performance:", lap_performance.shape)
print("Pit events:", pit_events.shape)
print("Laps:", laps.shape)

print("\nLap-performance columns:")
print(lap_performance.columns.tolist())

print("\nPit-event columns:")
print(pit_events.columns.tolist())

print("\nLap columns:")
print(laps.columns.tolist())

Lap performance: (962, 19)
Pit events: (45, 13)
Laps: (962, 16)

Lap-performance columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age']

Pit-event columns:
['driver_number', 'decision_lap', 'position_before', 'compound_before', 'tyre_age_before', 'rainfall', 'grip_condition', 'next_recorded_lap', 'position_next_recorded_lap', 'compound_after', 'tyre_age_after', 'next_is_pit_out', 'position_change_next_recorded_lap']

Lap columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed']


In [ ]:
# Notebook 11, Cell 24
# Match each recorded pit event to the driver's lap-performance data.
# This is an inspection and baseline-building step, not yet a validated
# pit-loss calculation.

import numpy as np
import pandas as pd

# Ensure keys and lap times have consistent numeric types.
for df in [lap_performance, laps]:
    df["driver_number"] = pd.to_numeric(
        df["driver_number"], errors="coerce"
    )
    df["lap_number"] = pd.to_numeric(
        df["lap_number"], errors="coerce"
    )
    df["lap_duration"] = pd.to_numeric(
        df["lap_duration"], errors="coerce"
    )

pit_events["driver_number"] = pd.to_numeric(
    pit_events["driver_number"], errors="coerce"
)
pit_events["decision_lap"] = pd.to_numeric(
    pit_events["decision_lap"], errors="coerce"
)
pit_events["next_recorded_lap"] = pd.to_numeric(
    pit_events["next_recorded_lap"], errors="coerce"
)

# Use lap_performance because it includes compound and tyre age.
lap_lookup = lap_performance[
    [
        "driver_number",
        "lap_number",
        "lap_duration",
        "compound",
        "tyre_age",
        "is_pit_out_lap",
        "stint_number",
    ]
].copy()

# One row per driver/lap is required for unambiguous matching.
duplicate_keys = lap_lookup.duplicated(
    ["driver_number", "lap_number"], keep=False
)

print("Duplicate driver/lap keys:", int(duplicate_keys.sum()))

if duplicate_keys.any():
    display(
        lap_lookup.loc[duplicate_keys]
        .sort_values(["driver_number", "lap_number"])
        .head(20)
    )
    raise ValueError(
        "Duplicate driver/lap keys found. Resolve these before matching "
        "pit events to lap times."
    )

# Attach the decision-lap observation.
decision_lookup = lap_lookup.rename(
    columns={
        "lap_number": "decision_lap",
        "lap_duration": "decision_lap_duration",
        "compound": "decision_lap_compound",
        "tyre_age": "decision_lap_tyre_age",
        "is_pit_out_lap": "decision_lap_is_pit_out",
        "stint_number": "decision_lap_stint",
    }
)

matched_pits = pit_events.merge(
    decision_lookup,
    on=["driver_number", "decision_lap"],
    how="left",
    validate="many_to_one",
)

# Attach the next recorded lap.
next_lookup = lap_lookup.rename(
    columns={
        "lap_number": "next_recorded_lap",
        "lap_duration": "next_lap_duration",
        "compound": "next_lap_compound",
        "tyre_age": "next_lap_tyre_age",
        "is_pit_out_lap": "next_lap_is_pit_out",
        "stint_number": "next_lap_stint",
    }
)

matched_pits = matched_pits.merge(
    next_lookup,
    on=["driver_number", "next_recorded_lap"],
    how="left",
    validate="many_to_one",
)

matched_pits["decision_lap_duration"] = pd.to_numeric(
    matched_pits["decision_lap_duration"], errors="coerce"
)
matched_pits["next_lap_duration"] = pd.to_numeric(
    matched_pits["next_lap_duration"], errors="coerce"
)

print("Pit events:", len(matched_pits))
print(
    "Matched decision laps:",
    int(matched_pits["decision_lap_duration"].notna().sum()),
)
print(
    "Matched next laps:",
    int(matched_pits["next_lap_duration"].notna().sum()),
)
print(
    "Marked pit-out laps:",
    int(matched_pits["next_lap_is_pit_out"].fillna(False).astype(bool).sum()),
)

display(
    matched_pits[
        [
            "driver_number",
            "decision_lap",
            "decision_lap_duration",
            "decision_lap_compound",
            "decision_lap_tyre_age",
            "next_recorded_lap",
            "next_lap_duration",
            "next_lap_compound",
            "next_lap_tyre_age",
            "next_lap_is_pit_out",
            "compound_after",
            "tyre_age_after",
        ]
    ].sort_values(["driver_number", "decision_lap"])
)

Duplicate driver/lap keys: 0
Pit events: 45
Matched decision laps: 45
Matched next laps: 45
Marked pit-out laps: 45


,driver_number,decision_lap,decision_lap_duration,decision_lap_compound,decision_lap_tyre_age,next_recorded_lap,next_lap_duration,next_lap_compound,next_lap_tyre_age,next_lap_is_pit_out,compound_after,tyre_age_after
0,1,26,101.491,MEDIUM,25.0,27.0,119.169,INTERMEDIATE,0.0,True,INTERMEDIATE,0.0
1,1,38,95.963,INTERMEDIATE,11.0,39.0,118.311,HARD,0.0,True,HARD,0.0
2,2,27,112.246,MEDIUM,26.0,28.0,125.981,INTERMEDIATE,0.0,True,INTERMEDIATE,0.0
3,2,38,98.811,INTERMEDIATE,10.0,39.0,116.256,SOFT,0.0,True,SOFT,0.0
4,3,26,104.813,MEDIUM,25.0,27.0,122.788,INTERMEDIATE,0.0,True,INTERMEDIATE,0.0
5,3,37,100.483,INTERMEDIATE,10.0,38.0,123.328,SOFT,0.0,True,SOFT,0.0
6,4,27,104.806,MEDIUM,26.0,28.0,120.915,INTERMEDIATE,0.0,True,INTERMEDIATE,0.0
7,4,39,96.746,INTERMEDIATE,11.0,40.0,114.840,SOFT,0.0,True,SOFT,0.0
8,11,19,101.868,HARD,18.0,20.0,119.150,INTERMEDIATE,0.0,True,INTERMEDIATE,0.0
9,11,28,105.224,INTERMEDIATE,8.0,29.0,124.385,INTERMEDIATE,0.0,True,INTERMEDIATE,0.0


In [ ]:

# Notebook 11, Cell 25
# Diagnostic baseline for pit-event analysis.
# Does NOT calculate or claim to measure pit loss.

import numpy as np
import pandas as pd

# Keep ordinary laps only, with valid lap times and tyre metadata.
baseline_laps = lap_performance.copy()

baseline_laps["lap_duration"] = pd.to_numeric(
    baseline_laps["lap_duration"], errors="coerce"
)
baseline_laps["tyre_age"] = pd.to_numeric(
    baseline_laps["tyre_age"], errors="coerce"
)

baseline_laps = baseline_laps.loc[
    baseline_laps["lap_duration"].notna()
    & baseline_laps["tyre_age"].notna()
    & ~baseline_laps["is_pit_out_lap"].fillna(False).astype(bool)
].copy()

# Broad sanity filter for clearly invalid lap times.
baseline_laps = baseline_laps.loc[
    baseline_laps["lap_duration"].between(50, 180)
].copy()

def estimate_local_baseline(event, laps_df, min_samples=5):
    """
    Estimate a local no-pit reference using the same driver and compound,
    with tyre age within +/- 2 laps.

    This is a diagnostic reference, not a causal counterfactual.
    """
    same_driver = laps_df["driver_number"].eq(event["driver_number"])
    same_compound = laps_df["compound"].eq(event["decision_lap_compound"])
    age_close = (
        (laps_df["tyre_age"] - event["decision_lap_tyre_age"]).abs() <= 2
    )

    # Use laps strictly before the pit decision to avoid future information.
    before_event = laps_df["lap_number"].lt(event["decision_lap"])

    candidates = laps_df.loc[
        same_driver & same_compound & age_close & before_event
    ]

    if len(candidates) < min_samples:
        return {
            "baseline_n": len(candidates),
            "baseline_median_seconds": np.nan,
            "baseline_supported": False,
        }

    return {
        "baseline_n": len(candidates),
        "baseline_median_seconds": float(
            candidates["lap_duration"].median()
        ),
        "baseline_supported": True,
    }

baseline_results = matched_pits.apply(
    lambda event: pd.Series(
        estimate_local_baseline(event, baseline_laps)
    ),
    axis=1,
)

pit_baseline_diagnostic = pd.concat(
    [
        matched_pits.reset_index(drop=True),
        baseline_results.reset_index(drop=True),
    ],
    axis=1,
)

print("Total pit events:", len(pit_baseline_diagnostic))
print(
    "Events with supported local baseline:",
    int(pit_baseline_diagnostic["baseline_supported"].sum()),
)
print(
    "Events without enough comparable laps:",
    int((~pit_baseline_diagnostic["baseline_supported"]).sum()),
)

display(
    pit_baseline_diagnostic[
        [
            "driver_number",
            "decision_lap",
            "decision_lap_duration",
            "decision_lap_compound",
            "decision_lap_tyre_age",
            "baseline_n",
            "baseline_median_seconds",
            "baseline_supported",
            "next_lap_duration",
            "next_lap_compound",
            "next_lap_is_pit_out",
        ]
    ].sort_values(["driver_number", "decision_lap"])
)


Total pit events: 45
Events with supported local baseline: 1
Events without enough comparable laps: 44


,driver_number,decision_lap,decision_lap_duration,decision_lap_compound,decision_lap_tyre_age,baseline_n,baseline_median_seconds,baseline_supported,next_lap_duration,next_lap_compound,next_lap_is_pit_out
0,1,26,101.491,MEDIUM,25.0,2,NaN,False,119.169,INTERMEDIATE,True
1,1,38,95.963,INTERMEDIATE,11.0,2,NaN,False,118.311,HARD,True
2,2,27,112.246,MEDIUM,26.0,2,NaN,False,125.981,INTERMEDIATE,True
3,2,38,98.811,INTERMEDIATE,10.0,2,NaN,False,116.256,SOFT,True
4,3,26,104.813,MEDIUM,25.0,2,NaN,False,122.788,INTERMEDIATE,True
5,3,37,100.483,INTERMEDIATE,10.0,2,NaN,False,123.328,SOFT,True
6,4,27,104.806,MEDIUM,26.0,2,NaN,False,120.915,INTERMEDIATE,True
7,4,39,96.746,INTERMEDIATE,11.0,2,NaN,False,114.840,SOFT,True
8,11,19,101.868,HARD,18.0,2,NaN,False,119.150,INTERMEDIATE,True
9,11,28,105.224,INTERMEDIATE,8.0,2,NaN,False,124.385,INTERMEDIATE,True


In [ ]:

# Notebook 11, Cell 26
# Compare increasingly broad no-stop pace baselines.
# Diagnostic only: these are not pit-loss measurements.

import numpy as np
import pandas as pd

# Add weather information if it is available in lap_performance.
# We do not invent weather values when they are absent.
weather_column_candidates = [
    c for c in ["weather_state", "weather_regime"]
    if c in lap_performance.columns
]

baseline_laps = lap_performance.copy()

baseline_laps["lap_number"] = pd.to_numeric(
    baseline_laps["lap_number"], errors="coerce"
)
baseline_laps["lap_duration"] = pd.to_numeric(
    baseline_laps["lap_duration"], errors="coerce"
)
baseline_laps["tyre_age"] = pd.to_numeric(
    baseline_laps["tyre_age"], errors="coerce"
)

baseline_laps = baseline_laps.loc[
    baseline_laps["lap_number"].notna()
    & baseline_laps["lap_duration"].between(50, 180)
    & baseline_laps["tyre_age"].notna()
    & ~baseline_laps["is_pit_out_lap"].fillna(False).astype(bool)
].copy()


def get_baseline_candidates(event, laps_df):
    earlier = laps_df.loc[
        laps_df["lap_number"] < event["decision_lap"]
    ].copy()

    same_driver_compound = earlier.loc[
        earlier["driver_number"].eq(event["driver_number"])
        & earlier["compound"].eq(event["decision_lap_compound"])
    ]

    nearby_age = same_driver_compound.loc[
        (same_driver_compound["tyre_age"]
         - event["decision_lap_tyre_age"]).abs() <= 2
    ]

    return {
        "same_driver_nearby_age": nearby_age,
        "same_driver_compound": same_driver_compound,
    }


diagnostics = []

for _, event in matched_pits.iterrows():
    candidates = get_baseline_candidates(event, baseline_laps)

    for method, subset in candidates.items():
        diagnostics.append({
            "driver_number": event["driver_number"],
            "decision_lap": event["decision_lap"],
            "compound": event["decision_lap_compound"],
            "method": method,
            "n": len(subset),
            "median_seconds": (
                float(subset["lap_duration"].median())
                if len(subset) else np.nan
            ),
        })

baseline_comparison = pd.DataFrame(diagnostics)

display(
    baseline_comparison.groupby("method").agg(
        pit_events=("decision_lap", "size"),
        supported_events=("median_seconds", "count"),
        median_candidate_count=("n", "median"),
    ).reset_index()
)

display(
    baseline_comparison.sort_values(
        ["driver_number", "decision_lap", "method"]
    ).head(20)
)


,method,pit_events,supported_events,median_candidate_count
0,same_driver_compound,45,44,11.0
1,same_driver_nearby_age,45,44,2.0


,driver_number,decision_lap,compound,method,n,median_seconds
1,1,26,MEDIUM,same_driver_compound,25,92.0380
0,1,26,MEDIUM,same_driver_nearby_age,2,94.5415
3,1,38,INTERMEDIATE,same_driver_compound,10,102.0615
2,1,38,INTERMEDIATE,same_driver_nearby_age,2,100.4135
5,2,27,MEDIUM,same_driver_compound,26,93.2290
4,2,27,MEDIUM,same_driver_nearby_age,2,100.4040
7,2,38,INTERMEDIATE,same_driver_compound,9,104.3320
6,2,38,INTERMEDIATE,same_driver_nearby_age,2,102.7860
9,3,26,MEDIUM,same_driver_compound,25,93.4290
8,3,26,MEDIUM,same_driver_nearby_age,2,97.6040


In [ ]:

# Notebook 11, Cell 27
# Diagnose whether weather-aware pace baselines have enough support.

from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed/9558")

# Reuse lap_performance already loaded in Notebook 11.
laps = lap_performance.copy()

# Locate the existing race-state dataset.
race_state_path = DATA_DIR / "race_state.parquet"

if not race_state_path.exists():
    raise FileNotFoundError(
        f"Could not find {race_state_path.resolve()}. "
        "Check DATA_DIR and the actual race_state.parquet location."
    )

race_state = pd.read_parquet(race_state_path)

print("Lap performance:", laps.shape)
print("Race state:", race_state.shape)
print("\nRace-state columns:")
print(race_state.columns.tolist())

# Standardize key types before joining.
for df in (laps, race_state):
    if "driver_number" in df.columns:
        df["driver_number"] = pd.to_numeric(
            df["driver_number"], errors="coerce"
        )
    if "lap_number" in df.columns:
        df["lap_number"] = pd.to_numeric(
            df["lap_number"], errors="coerce"
        )

# Identify the join key and weather fields available.
weather_candidates = [
    c for c in [
        "weather_state", "weather_regime", "rainfall",
        "grip_condition", "track_status"
    ]
    if c in race_state.columns
]

print("\nPotential weather/track fields:", weather_candidates)

# Join on driver + lap if both keys exist in race_state.
# Otherwise, stop rather than risk a many-to-many or incorrect merge.
join_keys = [
    c for c in ["driver_number", "lap_number"]
    if c in laps.columns and c in race_state.columns
]

if len(join_keys) != 2:
    raise ValueError(
        "race_state does not contain both driver_number and lap_number. "
        "Inspect its columns above and choose a valid join key before merging."
    )

# Keep only the join keys and relevant contextual columns.
context_cols = list(dict.fromkeys(join_keys + weather_candidates))
context = race_state[context_cols].copy()

# A driver-lap join must be unique on the context side.
duplicate_context_keys = context.duplicated(join_keys, keep=False).sum()
print("\nDuplicate race-state driver/lap rows:", duplicate_context_keys)

if duplicate_context_keys:
    raise ValueError(
        "race_state has duplicate driver/lap keys. Do not merge yet: "
        "resolve duplicates or use a time-aware merge based on timestamps."
    )

laps["lap_duration"] = pd.to_numeric(
    laps["lap_duration"], errors="coerce"
)
laps["tyre_age"] = pd.to_numeric(
    laps["tyre_age"], errors="coerce"
)

# Exclude pit-out laps and implausible lap durations from pace baselines.
laps["is_pit_out_lap"] = (
    laps["is_pit_out_lap"].fillna(False).astype(bool)
)

pace_laps = laps.loc[
    ~laps["is_pit_out_lap"]
    & laps["lap_duration"].between(50, 180)
    & laps["tyre_age"].notna()
].copy()

pace_laps = pace_laps.merge(
    context,
    on=join_keys,
    how="left",
    validate="many_to_one"
)

print("\nEligible normal laps:", len(pace_laps))
print("Weather/track field missingness:")
for col in weather_candidates:
    print(f"  {col}: {pace_laps[col].isna().mean():.1%} missing")

# Build a support table using driver, compound, weather/track state.
# We do not combine distinct weather states into one baseline.
group_cols = ["driver_number", "compound"] + weather_candidates

if not weather_candidates:
    print(
        "\nNo recognized weather/track fields were found. "
        "This is only a driver/compound baseline, not weather-aware."
    )

supported = (
    pace_laps.groupby(group_cols, dropna=False)
    .agg(
        n_laps=("lap_duration", "size"),
        median_lap_seconds=("lap_duration", "median"),
        min_tyre_age=("tyre_age", "min"),
        max_tyre_age=("tyre_age", "max"),
    )
    .reset_index()
    .sort_values("n_laps", ascending=False)
)

supported["support_status"] = np.select(
    [
        supported["n_laps"] >= 10,
        supported["n_laps"] >= 5,
    ],
    [
        "usable for diagnostic baseline",
        "weak support",
    ],
    default="insufficient support"
)

print("\nBaseline support summary:")
display(
    supported["support_status"]
    .value_counts()
    .rename_axis("support_status")
    .reset_index(name="groups")
)

print("\nLargest groups:")
display(supported.head(25))

print(
    "\nImportant: these are descriptive baselines, not causal pit-loss "
    "estimates. Weather and tyre-age overlap still need to be checked."
)
